# Entrenamiento en la nube (Kaggle / Colab)

Ejecuta un experimento de segmentación del visor de punta a punta: clona el repositorio, descarga y prepara los datos, entrena y empaqueta los resultados (`models/` y `experiments/`) en un zip para descargar.

**Kaggle:** activar *GPU* e *Internet* en *Settings*, cargar el Secret `ROBOFLOW_API_KEY` (*Add-ons → Secrets*) y ejecutar con *Save Version → Save & Run All* para que corra en segundo plano.

**Colab:** *Entorno de ejecución → GPU* y cargar el Secret `ROBOFLOW_API_KEY`.

In [ ]:
# Experimentos a ejecutar, en orden (ver experiments/README.md).
# dataset: "toloka" (solo Toloka) o "combined" (Toloka + yfzw4 con visor derivado).
# Cada experimento se empaqueta al terminar, así un corte por tiempo no pierde los anteriores.
EXPERIMENTS = [
    {"name": "toloka_yfzw4_yolov8n_seg", "dataset": "combined", "model": "yolov8n-seg.pt",
     "imgsz": 640, "epochs": 40, "patience": 10, "batch": 16},        # B
    {"name": "toloka_yfzw4_yolov8n_seg_1024", "dataset": "combined", "model": "yolov8n-seg.pt",
     "imgsz": 1024, "epochs": 40, "patience": 10, "batch": 16},       # C
]

In [ ]:
# Configuración del entorno: Kaggle, Colab o local.
import os
import subprocess
import sys
from pathlib import Path

IN_KAGGLE = Path("/kaggle/working").exists()
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/martinlacheski/ceia-vpc-2.git"


def run(cmd: list[str]) -> None:
    """Ejecuta un comando mostrando su salida y falla si termina con error."""
    print("$", " ".join(cmd))
    subprocess.run(cmd, check=True)


if IN_KAGGLE or IN_COLAB:
    os.chdir("/kaggle/working" if IN_KAGGLE else "/content")
    if not Path("ceia-vpc-2").exists():
        run(["git", "clone", "-q", REPO_URL])
    os.chdir("ceia-vpc-2")
    # Solo lo necesario para entrenar: PyTorch con CUDA ya viene instalado.
    run([sys.executable, "-m", "pip", "install", "-q", "ultralytics", "roboflow", "kagglehub", "opencv-python-headless"])

    try:
        if IN_KAGGLE:
            from kaggle_secrets import UserSecretsClient
            os.environ["ROBOFLOW_API_KEY"] = UserSecretsClient().get_secret("ROBOFLOW_API_KEY")
        else:
            from google.colab import userdata
            os.environ["ROBOFLOW_API_KEY"] = userdata.get("ROBOFLOW_API_KEY")
    except Exception as exc:
        print("Aviso: no se pudo leer el Secret ROBOFLOW_API_KEY:", exc)
elif Path.cwd().name == "notebooks":
    os.chdir("..")

import torch
print("Directorio:", Path.cwd(), "| Kaggle:", IN_KAGGLE, "| Colab:", IN_COLAB)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no disponible")

In [ ]:
# Descarga y preparación de los datos (solo los que piden los experimentos).
datasets = {e["dataset"] for e in EXPERIMENTS}
run([sys.executable, "scripts/download_data.py"] + ([] if "combined" in datasets else ["--only", "toloka"]))
run([sys.executable, "scripts/prepare_toloka.py"])
if "combined" in datasets:
    run([sys.executable, "scripts/prepare_seg_combined.py"])

DATA = {
    "toloka": "data/processed/toloka_seg/data.yaml",
    "combined": "data/processed/seg_toloka_yfzw4/data.yaml",
}

In [ ]:
# Entrenamiento, evaluación y empaquetado de cada experimento.
import shutil

out_dir = Path("/kaggle/working") if IN_KAGGLE else Path.cwd()


def package(name: str) -> str:
    """Arma <name>.zip con models/<name>.pt y experiments/<name>/ (en Kaggle: pestaña Output)."""
    staging = Path("runs/package") / name
    shutil.rmtree(staging, ignore_errors=True)
    (staging / "models").mkdir(parents=True)
    shutil.copy2(f"models/{name}.pt", staging / "models")
    shutil.copytree(f"experiments/{name}", staging / "experiments" / name)
    return shutil.make_archive(str(out_dir / name), "zip", root_dir=staging)


for exp in EXPERIMENTS:
    print(f"\n===== {exp['name']} =====")
    run([sys.executable, "scripts/train_yolo.py", "--task", "segment", "--model", exp["model"],
         "--data", DATA[exp["dataset"]], "--name", exp["name"], "--imgsz", str(exp["imgsz"]),
         "--epochs", str(exp["epochs"]), "--patience", str(exp["patience"]), "--batch", str(exp["batch"])])
    run([sys.executable, "scripts/evaluate_seg.py", "--model", f"models/{exp['name']}.pt",
         "--imgsz", str(exp["imgsz"])])
    print("Resultados:", package(exp["name"]))